# Mini-projeto 1 - Fase 2: CNN para classificação do CIFAR-10

Continuidade da Fase 1 (MLP, ver `../fase1-mlp/`). A lógica reutilizável (modelo, dados, treino, métricas, checkpointing) vive no pacote `cnn_cifar10` em `../src/`, seguindo exatamente o mesmo padrão da Fase 1 — o notebook fica focado em **definir experimentos e reportar resultados**, não em implementação.

**Integrantes do grupo:** _preencher aqui (nome de todos)_

O que este notebook cobre (conforme o enunciado do mini-projeto):
- Treino de uma CNN no CIFAR-10 com hiperparâmetros configuráveis (nº/tamanho de filtros, kernel size, stride, padding, pooling, dropout, taxa de aprendizagem, além dos já cobertos na Fase 1: ativação, otimizador, função de erro).
- Métricas por classe (acurácia) e globais (acurácia, precision, recall, f1).
- Comparação direta com o melhor resultado do MLP (Fase 1: ensemble `final` = 0.6135 de acurácia) — ver `../../fase1-mlp/README.md`.
- Cada execução de treino é salva automaticamente em `../results/` (pesos + config + métricas + histórico) — ver `../src/cnn_cifar10/checkpointing.py`.

**Recomendação forte: rode este notebook no Google Colab com GPU** (Ambiente de execução > Alterar tipo de ambiente de execução > GPU). CNN é bem mais lenta que MLP em CPU, e o ganho de GPU aqui é de 10-50x+.

## 0. Setup do ambiente

- **Local**: rode a partir de um ambiente onde o pacote já foi instalado (`pip install -e .` na pasta `fase2-cnn/`).
- **Google Colab**: a célula abaixo clona o repositório e instala o pacote automaticamente. O repositório é **privado**, então precisa de um GitHub Personal Access Token (PAT) — ver instruções abaixo, só precisa configurar uma vez.

### Gerar o token (uma vez só)

1. No GitHub: `Settings > Developer settings > Personal access tokens > Fine-grained tokens > Generate new token`.
2. Repository access: `Only select repositories` > `redes-neurais`.
3. Permissions: `Contents` = `Read-only` (só precisa ler/clonar, não escrever).
4. Defina uma expiração (ex.: 90 dias) e gere o token — copie o valor (`github_pat_...`), ele só aparece uma vez.

### Guardar no Colab (uma vez por navegador/conta)

1. No Colab, clique no ícone de chave (🔑 **Secrets**) na barra lateral esquerda.
2. `Add new secret` → nome `GITHUB_TOKEN`, valor = o token copiado acima.
3. Ative o toggle "Notebook access" para este notebook.

A célula abaixo lê o secret automaticamente; se não encontrar, pede o token via prompt (não fica salvo em lugar nenhum do notebook).

In [ ]:
#@title Setup (Colab ou local)
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import shutil

    BRANCH = "feat/cnn"  #@param {type:"string"}
    # Ajuste para "main" (ou o branch que estiver usando) quando o trabalho da
    # Fase 2 for mesclado - não precisa editar mais nada além desta linha.
    REPO_PATH = "github.com/jpbezerra/redes-neurais.git"
    REPO_DIR = Path("/content/redes-neurais")

    # Se uma tentativa anterior de clone falhou no meio (ex.: token errado),
    # a pasta pode existir mas sem ser um repositorio git valido - nesse caso
    # apagamos e clonamos de novo em vez de só tentar "git pull" nela.
    if REPO_DIR.exists() and not (REPO_DIR / ".git").exists():
        shutil.rmtree(REPO_DIR)

    if not REPO_DIR.exists():
        token = None
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
        if not token:
            import getpass
            token = getpass.getpass("Repositorio privado - cole seu GitHub Personal Access Token: ")
        clone_url = f"https://{token}@{REPO_PATH}"
        # o token fica salvo em .git/config só dentro desta VM efêmera do Colab
        # (destruída ao fim da sessão) - necessário para o "git pull" funcionar
        # de novo mais tarde na mesma sessão, sem pedir o token de novo.
        !git clone -q -b {BRANCH} {clone_url} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull -q origin {BRANCH}

    PROJECT_ROOT = REPO_DIR / "miniprojeto" / "fase2-cnn"
    assert (PROJECT_ROOT / "pyproject.toml").exists(), (
        f"Clone parece ter falhado (pyproject.toml nao encontrado em {PROJECT_ROOT}). "
        f"Confira: (1) BRANCH='{BRANCH}' e o branch certo, (2) o token em Secrets "
        "(GITHUB_TOKEN) esta valido - depois 'Ambiente de execucao > Reiniciar sessao' "
        "e rode esta celula de novo."
    )
    %pip install -q -e {PROJECT_ROOT}
else:
    PROJECT_ROOT = Path.cwd().parent  # notebooks/ -> fase2-cnn/

# Garante que o pacote seja importavel nesta sessao mesmo se o "pip install -e"
# editable nao for reconhecido pelo kernel ja em execucao (comum no Colab:
# %pip install atualiza o site-packages, mas o processo Python atual as vezes
# so releria o sys.path apos reiniciar o runtime) - adicionar direto e mais
# robusto do que depender de reiniciar a sessao toda vez.
sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
#@title Imports
import torch
import matplotlib.pyplot as plt
import pandas as pd

from cnn_cifar10.config import ExperimentConfig
from cnn_cifar10.data import get_dataloaders, CLASSES
from cnn_cifar10.train import fit, fit_or_load
from cnn_cifar10.checkpointing import load_all_metadata
from cnn_cifar10.utils import set_seed, get_device

In [ ]:
#@title Device
device = get_device()
print("Usando dispositivo:", device)
if device.type == "cpu":
    print("Aviso: sem GPU disponível. CNN em CPU é bem mais lenta — considere rodar no Colab.")

# No Colab/Linux, num_workers > 0 acelera o carregamento com augmentation
# (no Windows local, exige if __name__ == '__main__': em scripts, então mantemos 0 lá).
NUM_WORKERS = 2 if IN_COLAB else 0

## 1. Experimento baseline

Arquitetura equivalente à do notebook de referência do professor (`temp/CIFAR10_with_CNNs.ipynb`, adaptação do LeNet-5): 2 blocos convolucionais (32 e 64 filtros, kernel 3x3, padding 1, stride 1) + max pooling 2x2 após cada bloco, seguidos de cabeça densa `120 -> 84 -> 10`. ReLU, Adam, entropia cruzada, sem regularização — ponto de partida para a busca guiada, do mesmo jeito que a Fase 1 partiu do baseline MLP `[64,128,64]`.

In [ ]:
baseline_config = ExperimentConfig(
    run_name="baseline",
    conv_channels=(32, 64),
    kernel_size=3,
    stride=1,
    padding=1,
    pool_size=2,
    fc_layers=(120, 84),
    activation="relu",
    optimizer="adam",
    loss="cross_entropy",
    learning_rate=1e-3,
    batch_size=32,
    num_epochs=40,
    patience=5,
    notes="Baseline equivalente ao notebook de referencia (2 conv + 2 pool + 3 fc), ponto de partida da busca guiada da Fase 2.",
    tags=["baseline"],
)

set_seed(baseline_config.seed)
train_loader, val_loader, test_loader = get_dataloaders(
    data_dir=DATA_DIR,
    batch_size=baseline_config.batch_size,
    val_fraction=baseline_config.val_fraction,
    seed=baseline_config.seed,
    num_workers=NUM_WORKERS,
    augment=baseline_config.augment,
    normalization=baseline_config.normalization,
)

result_baseline = fit_or_load(
    baseline_config, train_loader, val_loader, test_loader, device,
    class_names=list(CLASSES), results_dir=RESULTS_DIR,
)
result_baseline["test_scores"]

## 2. Leva 1 — variações isoladas (uma alavanca por vez)

Mesmo espírito da leva 1 do MLP: cada configuração muda **um** hiperparâmetro
em relação ao baseline, para medir o efeito isolado antes de combinar
vencedores em rodadas sucessivas (busca gulosa, como nas levas 2-8 do MLP).
Cobre todos os parâmetros pedidos no enunciado da Fase 2: tamanho da rede,
kernel size, stride, padding, dropout, pooling e taxa de aprendizagem — mais
batch norm e augmentation como bônus (mesma cobertura extra feita no MLP).

`num_epochs=30`/`patience=5` (menor que o baseline) para essa leva exploratória
rodar mais rápido — se algum candidato for promissor, pode retreinar depois
com mais épocas.

In [ ]:
candidate_configs = [
    ExperimentConfig(
        run_name="kernel_5",
        conv_channels=(32, 64), kernel_size=5, stride=1, padding=2, pool_size=2,
        fc_layers=(120, 84), num_epochs=30, patience=5,
        notes="Kernel 5x5 em vez de 3x3 (padding=2 mantem o tamanho espacial 'same').",
    ),
    ExperimentConfig(
        run_name="stride_2_no_pool",
        conv_channels=(32, 64), kernel_size=3, stride=2, padding=1, pool_size=1,
        fc_layers=(120, 84), num_epochs=30, patience=5,
        notes="Substitui o pooling por stride=2 na propria convolucao (pool_size=1 = sem pooling extra).",
    ),
    ExperimentConfig(
        run_name="padding_0",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=0, pool_size=2,
        fc_layers=(120, 84), num_epochs=30, patience=5,
        notes="Convolucao 'valid' (sem padding) em vez de 'same' (padding=1) do baseline.",
    ),
    ExperimentConfig(
        run_name="pool_4",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=4,
        fc_layers=(120, 84), num_epochs=30, patience=5,
        notes="Janela de pooling maior (4x4 em vez de 2x2) apos cada bloco convolucional.",
    ),
    ExperimentConfig(
        run_name="deeper_3conv",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), num_epochs=30, patience=5,
        notes="Rede maior: 3 blocos convolucionais (32/64/128 filtros) em vez de 2.",
    ),
    ExperimentConfig(
        run_name="dropout_03",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), dropout=0.3, num_epochs=30, patience=5,
        notes="Dropout 0.3 (Dropout2d nos blocos conv + Dropout na cabeca densa) sobre o baseline.",
    ),
    ExperimentConfig(
        run_name="batch_norm",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), batch_norm=True, num_epochs=30, patience=5,
        notes="Adiciona BatchNorm2d apos cada convolucao, sobre o baseline.",
    ),
    ExperimentConfig(
        run_name="lr_low",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), learning_rate=5e-4, num_epochs=30, patience=5,
        notes="Learning rate 2x menor que o baseline (1e-3 -> 5e-4).",
    ),
    ExperimentConfig(
        run_name="lr_high",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), learning_rate=5e-3, num_epochs=30, patience=5,
        notes="Learning rate 5x maior que o baseline (1e-3 -> 5e-3).",
    ),
    ExperimentConfig(
        run_name="augment",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, num_epochs=30, patience=5,
        notes="Data augmentation (crop+flip) no treino, mesma transform da Fase 1.",
    ),
    # Adicione outras variacoes conforme os experimentos forem sendo decididos.
]

In [ ]:
experiment_results = {baseline_config.run_name: result_baseline}
for config in candidate_configs:
    if config.run_name in experiment_results:
        continue
    set_seed(config.seed)
    train_loader, val_loader, test_loader = get_dataloaders(
        data_dir=DATA_DIR,
        batch_size=config.batch_size,
        val_fraction=config.val_fraction,
        seed=config.seed,
        num_workers=NUM_WORKERS,
        augment=config.augment,
        normalization=config.normalization,
    )
    experiment_results[config.run_name] = fit_or_load(
        config, train_loader, val_loader, test_loader, device,
        class_names=list(CLASSES), results_dir=RESULTS_DIR,
    )

In [ ]:
#@title Tabela comparativa dos experimentos
comparison = pd.DataFrame(
    {name: r["test_scores"] for name, r in experiment_results.items()}
).T.sort_values("accuracy", ascending=False)
comparison

## 3. Próximas rodadas

Depois de ver os resultados da leva 1 acima, combine os hiperparâmetros
vencedores em rodadas sucessivas (busca gulosa), como nas levas 2-8 do MLP —
cada rodada informada pelo resultado da anterior. Se as rodadas ficarem
grandes/lentas demais para caber numa célula, considere mover para um
`scripts/run_experiments.py` (ver `../fase1-mlp/scripts/run_experiments.py`
como referência de estrutura).

In [ ]:
#@title Comparar todas as execuções salvas
df = load_all_metadata(RESULTS_DIR)
if not df.empty:
    cols = [c for c in ["run_name", "metrics.test_accuracy", "metrics.test_f1_score", "metrics.epochs_trained"] if c in df.columns]
    display(df[cols].sort_values("metrics.test_accuracy", ascending=False) if cols else df)
else:
    print("Nenhuma execucao salva ainda em", RESULTS_DIR)

## 4. Baixar `results/` para trazer de volta ao repositório local

**Só necessário no Colab** (local já grava direto em `../results/`). Os
resultados salvos em `/content/redes-neurais/.../results/` somem quando a
sessão do Colab reinicia — rode esta célula ao final de cada sessão de
experimentos para não perder o trabalho. Ela gera um `.zip` e baixa para a
pasta de Downloads do seu computador; depois é só extrair por cima da pasta
`results/` local (ou pedir para o Claude fazer isso) e dar commit/push.

In [ ]:
#@title Baixar results/ (Colab)
if IN_COLAB:
    import shutil
    from google.colab import files

    zip_path = shutil.make_archive("/content/results_cnn", "zip", str(RESULTS_DIR))
    print("Gerado:", zip_path)
    files.download(zip_path)
else:
    print("Local: results/ ja esta em", RESULTS_DIR, "- nao precisa baixar nada.")